<a href="https://colab.research.google.com/github/vanezp01/datalabs/blob/main/digitaltwinheart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ============================================================
# DIGITAL TWIN HEART DEMO (PLOTLY VERSION)
# ============================================================

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import random

# ============================================================
# ECG GENERATOR
# ============================================================

def generate_ecg(t, bpm):

    frequency = bpm / 60

    ecg = (
        0.05 * np.sin(2 * np.pi * frequency * t)
        + 0.1 * np.sin(2 * np.pi * frequency * 2 * t)
    )

    spike = (t * frequency) % 1

    if 0.48 < spike < 0.52:
        ecg += 1.5 * np.exp(-((spike - 0.5) ** 2) / 0.0005)

    ecg += np.random.normal(0, 0.02)

    return ecg

# ============================================================
# HEART SHAPE
# ============================================================

def create_heart(scale=1.0, rotation=0):

    t = np.linspace(0, 2*np.pi, 400)

    x = scale * 16 * np.sin(t)**3

    y = scale * (
        13*np.cos(t)
        - 5*np.cos(2*t)
        - 2*np.cos(3*t)
        - np.cos(4*t)
    )

    # Rotation
    xr = x*np.cos(rotation) - y*np.sin(rotation)
    yr = x*np.sin(rotation) + y*np.cos(rotation)

    return xr, yr

# ============================================================
# FAKE AI PREDICTION
# ============================================================

def fake_ai_risk(history):

    avg = np.mean(history)

    variability = np.std(history)

    risk = 0

    if avg > 95:
        risk += 40

    if variability > 10:
        risk += 35

    risk += random.randint(0, 15)

    risk = min(risk, 100)

    return risk

# ============================================================
# GENERATE FRAMES
# ============================================================

frames = []

WINDOW = 200

ecg_history = [0] * WINDOW

heart_history = [75] * 20

time_counter = 0

for frame in range(150):

    time_counter += 0.03

    # Simulated BPM
    bpm = 75 + random.randint(-5, 5)

    # Random cardiac event
    if random.random() < 0.04:
        bpm += random.randint(20, 45)

    heart_history.append(bpm)

    if len(heart_history) > 20:
        heart_history.pop(0)

    # ECG
    ecg = generate_ecg(time_counter, bpm)

    ecg_history.append(ecg)

    if len(ecg_history) > WINDOW:
        ecg_history.pop(0)

    # Fake AI
    risk = fake_ai_risk(heart_history)

    if risk < 30:
        status = "STABLE"
        color = "lime"

    elif risk < 60:
        status = "WARNING"
        color = "yellow"

    else:
        status = "CRITICAL"
        color = "red"

    # Animated Heart
    pulse = 1 + 0.05 * np.sin(frame * 0.3)

    rotation = frame * 0.05

    hx, hy = create_heart(
        scale=pulse,
        rotation=rotation
    )

    # ========================================================
    # FRAME
    # ========================================================

    frames.append(

        go.Frame(

            data=[

                # ECG
                go.Scatter(
                    y=ecg_history,
                    mode='lines',
                    line=dict(
                        color=color,
                        width=3
                    ),
                    name="ECG"
                ),

                # Heart
                go.Scatter(
                    x=hx,
                    y=hy,
                    mode='lines',
                    fill='toself',
                    line=dict(
                        color=color,
                        width=4
                    ),
                    name="Heart"
                )

            ],

            layout=go.Layout(

                title=dict(
                    text=(
                        f"<b>DIGITAL TWIN HEART MONITOR</b><br>"
                        f"Heart Rate: {bpm} BPM | "
                        f"AI Risk: {risk}% | "
                        f"{status}"
                    ),
                    font=dict(size=22)
                )

            )

        )

    )

# ============================================================
# INITIAL FIGURE
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Live ECG Signal",
        "Digital Heart Twin"
    )
)

# ECG
fig.add_trace(

    go.Scatter(
        y=ecg_history,
        mode='lines',
        line=dict(
            color='lime',
            width=3
        )
    ),

    row=1,
    col=1
)

# Heart
hx, hy = create_heart()

fig.add_trace(

    go.Scatter(
        x=hx,
        y=hy,
        mode='lines',
        fill='toself',
        line=dict(
            color='red',
            width=4
        )
    ),

    row=1,
    col=2
)

# ============================================================
# LAYOUT
# ============================================================

fig.update_layout(

    template="plotly_dark",

    width=1200,
    height=600,

    title="<b>AI-Powered Digital Twin Demo</b>",

    showlegend=False,

    updatemenus=[

        dict(

            type="buttons",

            buttons=[

                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        {
                            "frame": {"duration": 60},
                            "fromcurrent": True
                        }
                    ]
                )

            ]

        )

    ]

)

# ECG styling
fig.update_xaxes(
    visible=False,
    row=1,
    col=1
)

fig.update_yaxes(
    range=[-2, 2],
    row=1,
    col=1
)

# Heart styling
fig.update_xaxes(
    visible=False,
    range=[-25, 25],
    row=1,
    col=2
)

fig.update_yaxes(
    visible=False,
    range=[-25, 25],
    scaleanchor="x",
    row=1,
    col=2
)

# ============================================================
# ATTACH FRAMES
# ============================================================

fig.frames = frames

# ============================================================
# SHOW
# ============================================================

fig.show()